# Mosaic Generation Pipeline — Multi-Slide
End-to-end: discover NDPI/XML pairs → load annotations → cluster → extract pairs → build mosaic → visualise.

In [ ]:
from util import (
    read_annotations_xml,
    cluster_annotations,
    read_region,
    make_segmentation_mask,
    make_content_mask,
    build_mosaic,
    apply_tissue_mask,
)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import pyvips
from pathlib import Path

In [ ]:
# --- Slide directory (must contain matched .ndpi + .xml pairs by stem) ---
SLIDE_DIR = Path('/home/donald/Desktop/shelter_server_donald/data/PDAC_Dimitri/annotations_regular_CODA')

# --- Parameters ---
LEVEL          = 1      # pyramid level to read from
CLUSTER_PAD    = 0      # padding (px, level-0) when clustering annotation bboxes
CANVAS_SIZE    = 10240*2  # mosaic canvas side length in pixels
GAP            = 0      # gap between placed regions (px)
SEED           = 67     # random seed for mosaic placement

# --- Mosaic placement parameters ---
PLACEMENT_CANDIDATES = 128   # ↑ more candidates → better positions found (was 32)
OVERLAP_PENALTY      = 0.8   # ↑ strongly avoid overlapping existing tiles (was 0.3)
GRID_CELL            = 8     # ↓ finer scoring grid → more accurate placement (was 16)
FILL_THRESHOLD       = 0.85  # ↑ keep placing until 85% filled (default was 0.5)

# --- Tissue Area (TA) parameters ---
TA_G_THRESH            = 190   # green-channel threshold: pixels with G < this are tissue
TA_SCALE               = 3     # downscale factor for TA detection (1 = no downscale)
TA_MIN_BG_PIXELS       = 500   # min background component size — smaller holes are filled as tissue
WHITESPACE_LABEL       = "Whitespace" # annotation label to force → class 0 (background)

# --- Plotting ---
SHOW_ANNOTATION_PLOTS = True   # set True to visualise each slide's annotations + clusters

In [ ]:
# --- Priority: index 0 = highest priority (painted last = wins) ---
# Note: WHITESPACE_LABEL is excluded from PRIORITY_ORDER — it is always mapped to 0
PRIORITY_ORDER = [
    WHITESPACE_LABEL,
    "Vasculature",
    "Muscle",
    "Nerve",
    "Fat",
    "TLS",
    "PDAC",
    "Duct",
    "Islet",
    "ECM",  # lowest priority = painted first = gets overpainted by all other labels
]

label_to_class = {lbl: len(PRIORITY_ORDER) - i for i, lbl in enumerate(PRIORITY_ORDER)}

print("label → class index (higher = on top):")
for lbl, idx in sorted(label_to_class.items(), key=lambda x: -x[1]):
    print(f"  {idx:2d}  {lbl}")

In [ ]:
# --- Discover NDPI / XML pairs ---
ndpi_files = {p.stem: p for p in SLIDE_DIR.glob("*.ndpi")}
xml_files  = {p.stem: p for p in SLIDE_DIR.glob("*.xml")}
stems      = sorted(ndpi_files.keys() & xml_files.keys())

print(f"Found {len(stems)} matched NDPI/XML pairs:")
for s in stems:
    print(f"  {s}")

In [ ]:
# --- Load annotations, cluster, and extract pairs from all slides ---
all_pairs         = []   # list of (img, mask, content_mask) triples
all_clusters_info = []

for stem in stems:
    pth_ndpi = str(ndpi_files[stem])
    pth_xml  = str(xml_files[stem])

    xml      = read_annotations_xml(pth_xml)
    clusters = cluster_annotations(xml, padding=CLUSTER_PAD)

    labels_found = sorted(set(a.label for a in xml))

    # --- Optional annotation visualisation ---
    if SHOW_ANNOTATION_PLOTS:
        slide  = pyvips.Image.new_from_file(pth_ndpi, access='sequential', level=0)
        W0, H0 = slide.width, slide.height
        colors      = plt.cm.tab10.colors
        label_color = {l: colors[i % len(colors)] for i, l in enumerate(labels_found)}

        fig, ax = plt.subplots(figsize=(7, 5))
        for a in xml:
            poly = plt.Polygon(a.coords, closed=True, fill=True, alpha=0.15,
                               facecolor=label_color[a.label], edgecolor=label_color[a.label], linewidth=0.4)
            ax.add_patch(poly)
        for i, bb in enumerate(clusters):
            rect = mpatches.Rectangle((bb.x_min, bb.y_min), bb.x_max - bb.x_min, bb.y_max - bb.y_min,
                                      linewidth=1.5, edgecolor='black', facecolor='none', linestyle='--')
            ax.add_patch(rect)
            ax.text(bb.x_min, bb.y_min - 30, f"#{i}", fontsize=6, color='black', va='bottom')
        ax.set_xlim(0, W0); ax.set_ylim(0, H0); ax.invert_yaxis(); ax.set_aspect('equal')
        ax.set_title(f"{stem} — {len(clusters)} clusters from {len(xml)} annotations")
        ax.legend(handles=[mpatches.Patch(color=label_color[l], label=l) for l in labels_found],
                  loc='upper right', fontsize=8)
        plt.tight_layout(); plt.show()

    # --- Extract pairs ---
    for bb in clusters:
        img     = read_region(pth_ndpi, bb, level=LEVEL)
        mask    = make_segmentation_mask(bb, xml, label_to_class, level=LEVEL, slide_path=pth_ndpi)
        content = make_content_mask(bb, xml, level=LEVEL, slide_path=pth_ndpi)
        mask    = apply_tissue_mask(mask, img,
                                    g_thresh=TA_G_THRESH,
                                    scale_factor=TA_SCALE,
                                    min_background_pixels=TA_MIN_BG_PIXELS)
        # Force any remaining WHITESPACE_LABEL pixels → 0 (belt-and-suspenders)
        whitespace_class = label_to_class.get(WHITESPACE_LABEL, None)
        if whitespace_class is not None and whitespace_class != 0:
            mask[mask == whitespace_class] = 0
        all_pairs.append((img, mask, content))
        all_clusters_info.append((stem, bb))

print(f"\nTotal pairs collected: {len(all_pairs)}")

In [ ]:
# --- Preview first pair: H&E | mask | content | overlay (for TA tuning) ---
img0, mask0, content0 = all_pairs[0]
stem0, bb0  = all_clusters_info[0]

num_classes = len(PRIORITY_ORDER) + 1
cmap_base   = plt.cm.get_cmap("tab10", num_classes)
colors_rgba = cmap_base(np.arange(num_classes))
colors_rgba[0] = [0, 0, 0, 1]
from matplotlib import colors as mcolors
pair_cmap = mcolors.ListedColormap(colors_rgba)

tissue_frac = (mask0 > 0).mean()

fig, axes = plt.subplots(1, 4, figsize=(24, 6))

axes[0].imshow(img0[..., :3])
axes[0].set_title(f"H&E  [{stem0}  #{0}]", fontsize=10)
axes[0].axis('off')

axes[1].imshow(content0, cmap='gray')
axes[1].set_title("Content mask (annotation extent)", fontsize=10)
axes[1].axis('off')

axes[2].imshow(mask0, cmap=pair_cmap, vmin=0, vmax=num_classes - 1)
axes[2].set_title(f"Segmentation mask  (tissue={tissue_frac*100:.1f}%)", fontsize=10)
axes[2].axis('off')

axes[3].imshow(img0[..., :3])
axes[3].imshow(mask0, cmap=pair_cmap, vmin=0, vmax=num_classes - 1, alpha=0.5)
axes[3].set_title("Overlay", fontsize=10)
axes[3].axis('off')

legend_handles = [mpatches.Patch(color=pair_cmap(0), label="0: background")]
for lbl, idx in sorted(label_to_class.items(), key=lambda x: x[1]):
    legend_handles.append(mpatches.Patch(color=pair_cmap(idx), label=f"{idx}: {lbl}"))
fig.legend(handles=legend_handles, loc='lower center', ncol=5, fontsize=9, frameon=True)
plt.suptitle(f"Pair #0 preview — TA_G_THRESH={TA_G_THRESH}  TA_SCALE={TA_SCALE}  TA_MIN_BG_PIXELS={TA_MIN_BG_PIXELS}", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# --- Build mosaic from all slides ---
he_mosaic, mask_mosaic = build_mosaic(
    all_pairs,
    canvas_size=CANVAS_SIZE,
    fill_threshold=FILL_THRESHOLD,
    gap=GAP,
    allow_rotation=True,
    placement_candidates=PLACEMENT_CANDIDATES,
    overlap_penalty=OVERLAP_PENALTY,
    grid_cell=GRID_CELL,
    rng=np.random.default_rng(SEED),
)

filled_frac = (mask_mosaic > 0).mean()
print(f"Canvas filled : {filled_frac * 100:.1f}%")
print(f"Canvas size   : {CANVAS_SIZE} × {CANVAS_SIZE} px")
print(f"Slides used   : {len(stems)}")

In [ ]:
filled_frac = (mask_mosaic > 0).mean()   # fraction of non-zero mask pixels
DISPLAY_SIZE = 1024
he_display   = cv2.resize(he_mosaic,   (DISPLAY_SIZE, DISPLAY_SIZE), interpolation=cv2.INTER_AREA)
mask_display = cv2.resize(mask_mosaic, (DISPLAY_SIZE, DISPLAY_SIZE), interpolation=cv2.INTER_NEAREST)

num_classes = len(PRIORITY_ORDER) + 1
cmap_base   = plt.cm.get_cmap("tab10", num_classes)
colors_rgba = cmap_base(np.arange(num_classes))
colors_rgba[0] = [0, 0, 0, 1]
from matplotlib import colors as mcolors
cmap = mcolors.ListedColormap(colors_rgba)

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
axes[0].imshow(he_display)
axes[0].set_title("H&E mosaic", fontsize=13)
axes[0].axis('off')

axes[1].imshow(mask_display, cmap=cmap, vmin=0, vmax=num_classes - 1)
axes[1].set_title("Mask overlay", fontsize=13)
axes[1].axis('off')

legend_handles = [mpatches.Patch(color=cmap(0), label="0: background")]
for lbl, idx in sorted(label_to_class.items(), key=lambda x: x[1]):
    legend_handles.append(mpatches.Patch(color=cmap(idx), label=f"{idx}: {lbl}"))
fig.legend(handles=legend_handles, loc='lower center', ncol=4, fontsize=10, frameon=True)
plt.suptitle(f"Mosaic — {filled_frac*100:.1f}% filled  |  {len(stems)} slides  |  seed={SEED}", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# save H&E mosaic and mask as .png files:
outpth = 'annotations/big_tile'

from PIL import Image
import os

Image.fromarray(he_mosaic).save(os.path.join(outpth, 'mosaic_he.png'))
Image.fromarray(mask_mosaic).save(os.path.join(outpth, 'mosaic_mask.png'))